In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

In [21]:
url = "https://raw.githubusercontent.com/AsaelP25/etl-data-pipeline-2510772022/refs/heads/main/data/raw/D_departamentos.csv"

In [22]:
df = pd.read_csv(url)

In [23]:
df.head()

,id_departamento,departamento,ubicacion
0,D10,TI,Edificio A
1,D11,Finanzas,Remoto
2,D12,RRHH,Edificio A
3,D13,Ventas,Edificio A
4,D14,Compras,Edificio A


Exploracion de archivos

In [24]:
df.shape

(9, 3)

In [25]:
df.columns

Index(['id_departamento', 'departamento', 'ubicacion'], dtype='object')

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_departamento  9 non-null      object
 1   departamento     9 non-null      object
 2   ubicacion        9 non-null      object
dtypes: object(3)
memory usage: 348.0+ bytes


In [27]:
df.isnull().sum()

,0
id_departamento,0
departamento,0
ubicacion,0


Limpieza de datos

In [60]:
departamentos = df.copy()

In [61]:
departamentos.columns = departamentos.columns.str.strip().str.lower()

In [62]:
for col in departamentos.select_dtypes(include='object').columns:
  departamentos[col] = departamentos[col].astype(str).str.strip()

In [90]:
departamentos = departamentos.replace(r'^\s*$',pd.NA, regex=True)

In [91]:
duplicados = df[df.duplicated(subset=['id_departamento'], keep=False)]

In [92]:
rejects_dup = duplicados.copy()

In [93]:
departamentos = df.drop_duplicates(subset=['id_departamento'], keep='first')

Separar datos válidos y rechazados

In [94]:
validos = (
    departamentos['id_departamento'].str.match(r'^D\d+$', na=False) &
    departamentos['departamento'].notna() &
    departamentos['ubicacion'].notna()
)

curated = departamentos[validos].copy()
rejects = pd.concat([
    departamentos[~validos],
    rejects_dup
])

In [96]:
curated = departamentos[validos].copy()
rejects = departamentos[~validos].copy()

Exportacion

In [97]:
curated.to_csv('/content/curated_departamentos.csv', index=False)
rejects.to_csv('/content/rejects_departamentos.csv', index=False)